<a href="https://colab.research.google.com/github/yungching-tsai/114-2ProgramingLanguage/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [39]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [55]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)
# genai.configure(api_key=api_key)

model_ID = 'gemini-2.5-flash'

In [56]:
response = client.models.generate_content(
    model = model_ID,
    contents="Say hi, and explain how AI works in a few words",
)
print(response.text)

Hi!

AI works by **learning from massive amounts of data** to find patterns, make predictions, or solve problems, much like humans learn from experience.


In [46]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Rs8SWFg2ffNJzmvOQb1ALV5D3ojTyjX6_SarTSE_yDE/edit?usp=sharing"
WORKSHEET_NAME = "成績單"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [47]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [61]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = model_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [49]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：100
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 100

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：81
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 81

請輸入科目（或輸入 'q' 停止）：地理
請輸入 地理 的成績：74
已記錄：日期: 2026-03-26, 科目: 地理, 成績: 74

請輸入科目（或輸入 'q' 停止）：q


In [50]:
new_grades

[['2026-03-26', '國文', 100], ['2026-03-26', '數學', 81], ['2026-03-26', '地理', 74]]

In [62]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份基於您提供的成績列表所做的簡單摘要與常見迷思整理，全程不涉及任何評價或判斷。\n\n---\n\n### **成績列表摘要**\n\n這份成績列表呈現了學生在 **2026年3月26日** 進行的三個科目的考試表現：\n\n*   **國文：100分**\n*   **數學：81分**\n*   **地理：74分**\n\n整體而言，學生的成績分佈介於74分至100分之間。其中，國文科目獲得滿分100分；數學科目得分為81分；地理科目得分為74分。所有科目成績均達到70分以上。\n\n### **關於成績的常見迷思整理**\n\n以下是人們在看待學生學業成績時，可能普遍存在的一些常見迷思，這些迷思可能導致對學生學習狀況的不全面理解：\n\n1.  **單次成績完全代表學生的能力或智力水平：**\n    *   **迷思：** 認為一次考試的分數就能全面衡量學生的智商、學習潛力或綜合能力。\n    *   **事實：** 成績往往是特定時間點、特定測驗範圍和形式下學習成果的反映，可能受到多種因素影響，如考試壓力、健康狀況、題目難度、學習策略，甚至當天的運氣。它無法涵蓋學生所有類型的能力。\n\n2.  **高分意味著無需改進，低分則代表徹底失敗：**\n    *   **迷思：** 將滿分視為學習的終點，而將較低的分數視為學習上的無可挽回的失敗。\n    *   **事實：** 即使是高分，也可能有進一步提升的空間（例如更深入的理解、更廣泛的應用），或者只是該次考試題目恰好符合學生擅長的部分。較低的分數則可以被視為指出需要更多關注、不同學習方法或額外支持的領域，是學習過程中的回饋，而非終結。\n\n3.  **分數是衡量學習成果的唯一或最全面指標：**\n    *   **迷思：** 堅信學生的分數越高，他們的學習成果就越好，且這是衡量學習成就的唯一標準。\n    *   **事實：** 學習成果涵蓋知識的理解、應用、批判性思考能力、解決問題的能力、創造力、溝通與合作技巧、學習態度等多元面向。成績僅是其中一種量化指標，未能完全體現學生在這些非量化領域的成長。\n\n4.  **直接比較不同學生的分數可以全面且公正地評估他們的學習進度或潛力：**\n    *   **迷思：** 認為只要看分數就能公平比較不同學生之間的優劣。\n    * 

In [63]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：99
已記錄：日期: 2026-03-26, 科目: 國文, 成績: 99

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：86
已記錄：日期: 2026-03-26, 科目: 數學, 成績: 86

請輸入科目（或輸入 'q' 停止）：歷史
請輸入 歷史 的成績：61
已記錄：日期: 2026-03-26, 科目: 歷史, 成績: 61

請輸入科目（或輸入 'q' 停止）：物理
請輸入 物理 的成績：68
已記錄：日期: 2026-03-26, 科目: 物理, 成績: 68

請輸入科目（或輸入 'q' 停止）：生物
請輸入 生物 的成績：85
已記錄：日期: 2026-03-26, 科目: 生物, 成績: 85

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程不對學生的表現進行任何評分或判斷：

---

### 成績列表摘要

這份成績列表記錄了該學生在 **2026年3月26日** 取得的五個科目分數。

*   **科目與分數分佈：**
    *   國文：99分
    *   數學：86分
    *   生物：85分
    *   物理：68分
    *   歷史：61分

*   **整體觀察：**
    *   最高分數為國文的99分。
    *   最低分數為歷史的61分。
    *   其他科目的分數介於68分至86分之間。
    *   若計算這五個科目的平均分數，約為79.8分。

---

### 常見迷思整理（關於學業成績）

以下是關於學業成績的一些常見迷思，旨在提供更全面的視角，而非針對特定分數進行判斷：

1.  **迷思一：分數是衡量學生全部能力的唯一標準。*